# Quantum Cryptanalysis on IBM Quantum Hardware — Reproduction Notebook
**ArXivist-generated** | Paper: [arXiv:2607.18340](https://arxiv.org/abs/2607.18340)

Walks through all five disclosed quantum attacks from the paper: Bernstein-Vazirani,
Grover (SPN key search), and Simon's algorithm applied to Even-Mansour, CBC-MAC
forgery, and 3-round Feistel.

> **Important**: the paper's headline real-hardware result (Even-Mansour N=6-10)
> depends on a technique the paper explicitly withholds (Section 5). This notebook
> runs the disclosed algorithms on a noiseless Qiskit Aer simulator, where they
> succeed by construction. See `README.md` for full detail.


In [2]:
import sys
sys.path.insert(0, "../src")
import random
import qiskit
print(f"Qiskit version: {qiskit.__version__}")


Qiskit version: 1.1.0


In [3]:
from quantum_cryptanalysis.backend.execution import BackendFactory
backend = BackendFactory("aer_simulator_noiseless").get_backend()
print("Backend ready:", backend)


Backend ready: AerSimulator('aer_simulator')


## Attack 1 — Bernstein-Vazirani (linear structure, 1 query)

$$f(x) = a \cdot x \pmod 2$$

A single quantum query plus a Hadamard transform recovers the secret $a$ directly.


In [4]:
from quantum_cryptanalysis.oracles.bernstein_vazirani import BVOracle
from quantum_cryptanalysis.algorithms.bv_circuit import BernsteinVazirani

secret = "10110101"  # 8-bit secret, matches paper's validated n=8
oracle = BVOracle().build_circuit(secret)
recovered = BernsteinVazirani().run(oracle, n=len(secret), backend=backend, shots=1)

print(f"true secret:  {secret}")
print(f"recovered:    {recovered}")
print(f"MATCH: {recovered == secret}  (single query, as claimed)")


true secret:  10110101
recovered:    10110101
MATCH: True  (single query, as claimed)


## Attack 2 — Grover (SPN key search, quadratic speedup)

Iteration count: $\frac{\pi}{4}\sqrt{2^n}$. At $n=8$: classical brute force needs
256 evaluations; Grover needs ~13 iterations (paper's Figure 4).

**Note**: the paper never specifies its SPN construction (S-box, rounds, key
schedule) -- `ToySPNCipher` below is a small, clearly-labeled stand-in.


In [5]:
from quantum_cryptanalysis.oracles.grover_spn import ToySPNCipher, GroverSPNOracle
from quantum_cryptanalysis.algorithms.grover_circuit import GroverSearch
import math

n_bits = 8
cipher = ToySPNCipher(n_bits)
rng = random.Random(0)
true_key = rng.randint(1, 2**n_bits - 1)

# pick a plaintext with a UNIQUE matching key (key->ciphertext at fixed
# plaintext need not be injective; see README "Verified findings" #4)
plaintext = None
for candidate_pt in range(2**n_bits):
    ct = cipher.encrypt(true_key, candidate_pt)
    if sum(1 for k in range(2**n_bits) if cipher.encrypt(k, candidate_pt) == ct) == 1:
        plaintext = candidate_pt
        break
ciphertext = cipher.encrypt(true_key, plaintext)

oracle = GroverSPNOracle().build_circuit(plaintext, ciphertext, n_bits)
recovered = GroverSearch().run(oracle, n=n_bits, backend=backend, shots=4096)

expected_iterations = round((math.pi/4) * math.sqrt(2**n_bits))
print(f"true key:      {format(true_key, f'0{n_bits}b')}")
print(f"recovered key: {format(recovered, f'0{n_bits}b')}")
print(f"MATCH: {recovered == true_key}")
print(f"Grover iterations used: {expected_iterations}  (paper's Figure 4 reports ~13 at n=8)")
print(f"Classical brute force would need up to {2**n_bits} evaluations")


true key:      11011001
recovered key: 11011001
MATCH: True
Grover iterations used: 13  (paper's Figure 4 reports ~13 at n=8)
Classical brute force would need up to 256 evaluations


## Attack 3 — Simon's algorithm: Even-Mansour period recovery

The Even-Mansour cipher $E_{k_1,k_2}(x) = P(x \oplus k_1) \oplus k_2$ reduces to a
hidden-period problem via $g(x) = P(x \oplus k_1) \oplus P(x)$, which has period
$k_1$ for ANY permutation $P$ (this is a general algebraic identity, verified
during implementation -- see README "Verified findings" #1).


In [6]:
from quantum_cryptanalysis.oracles.simon_even_mansour import EvenMansourOracle, find_valid_permutation
from quantum_cryptanalysis.algorithms.simon_circuit import SimonAlgorithm

simon = SimonAlgorithm()
n = 5  # paper's clean/differential-Simon regime is n<=5
rng = random.Random(1)
k1 = rng.randint(1, 2**n - 1)
perm = find_valid_permutation(n, k1, rng)

oracle = EvenMansourOracle().build_circuit(k1=k1, permutation=perm, n=n)
measurements = simon.collect_measurements(oracle, n=n, backend=backend, shots=500)
s = simon.solve_period(measurements, n=n)

print(f"true k1:    {format(k1, f'0{n}b')}")
print(f"recovered:  {s}")
print(f"MATCH: {s == format(k1, f'0{n}b')}  (clean/rank-1, matching paper's n<=5 regime)")


true k1:    00101
recovered:  00101
MATCH: True  (clean/rank-1, matching paper's n<=5 regime)


### What about n=6-10 (the paper's headline claim)?

The paper reports "hybrid" recovery here using its withheld technique. We can
demonstrate what happens WITHOUT that technique -- clean recovery degrades under
noise, and our best-effort substitute ranker (NOT the paper's real method) is
shown for illustration only.


In [7]:
from quantum_cryptanalysis.postprocessing.hybrid_ranking import TopKHybridRanker

n = 6
rng = random.Random(2)
k1 = rng.randint(1, 2**n - 1)
perm = find_valid_permutation(n, k1, rng)
oracle = EvenMansourOracle().build_circuit(k1=k1, permutation=perm, n=n)

noisy_backend = BackendFactory("aer_simulator_noisy").get_backend()
measurements = simon.collect_measurements(oracle, n=n, backend=noisy_backend, shots=500)

s_clean = simon.solve_period(measurements, n=n)
print(f"Clean linear-solve under noise: {s_clean}  (often fails/wrong -- expected under noise)")

ranker = TopKHybridRanker()
result = ranker.rank_candidates(measurements, n=n, true_key=k1, top_k=16)
print(f"True key rank (our labeled SUBSTITUTE ranker): {result['rank']} / {result['total_candidates']}")
print("This is NOT a reproduction of the paper's withheld technique -- see README.")


Clean linear-solve under noise: None  (often fails/wrong -- expected under noise)
True key rank (our labeled SUBSTITUTE ranker): 62 / 63
This is NOT a reproduction of the paper's withheld technique -- see README.


## Attack 4 — Simon's algorithm: CBC-MAC forgery

**Verified discrepancy**: the paper states period $s = E_k(a) \oplus E_k(b)$ for
$f(x) = E_k(x \oplus c \cdot a) \oplus E_k(x \oplus c \cdot b)$. Testing shows this
oracle instead satisfies Simon's promise with period $s = a \oplus b$ (the
*input*-space difference) -- see `oracles/simon_cbc_mac.py` for the full derivation.


In [8]:
from quantum_cryptanalysis.oracles.simon_cbc_mac import CBCMACForgeryOracle

n = 4
rng = random.Random(3)
a = rng.randint(1, 2**n - 1)
b = 0
perm = find_valid_permutation(n, a, rng)
def block_cipher(key, pt):
    return perm[pt]

true_s = a ^ b
oracle = CBCMACForgeryOracle().build_circuit(block_cipher, 0, a, b, n)
measurements = simon.collect_measurements(oracle, n=n, backend=backend, shots=500)
s = simon.solve_period(measurements, n=n)

print(f"true s (=a^b):  {format(true_s, f'0{n}b')}")
print(f"recovered:      {s}")
print(f"MATCH: {s == format(true_s, f'0{n}b')}")


true s (=a^b):  0100
recovered:      0100
MATCH: True


## Attack 5 — Simon's algorithm: 3-round Feistel (DES-family)

$$f(b,x) = F_2(x \oplus F_1(\alpha_b)), \quad s=(1,\gamma), \quad \gamma=F_1(\alpha_0)\oplus F_1(\alpha_1)$$

Clean (rank-1) if $F_2$ is a permutation. The paper stresses: placing the variable
on the left (x) and the constant on the right ($\alpha_b$) is essential.


In [9]:
from quantum_cryptanalysis.oracles.simon_feistel import FeistelOracle

m = 3  # half-block; paper's "block size 2m" = 6 here, matching its validated block-6 hardware size
rng = random.Random(4)
perm1 = list(range(2**m)); rng.shuffle(perm1)
perm2 = list(range(2**m)); rng.shuffle(perm2)  # F2 must be a permutation
f1, f2 = (lambda a: perm1[a]), (lambda a: perm2[a])
alpha0 = rng.randint(0, 2**m - 1)
alpha1 = rng.randint(0, 2**m - 1)
while alpha1 == alpha0:
    alpha1 = rng.randint(0, 2**m - 1)
gamma = f1(alpha0) ^ f1(alpha1)
true_s = format(1 | (gamma << 1), f'0{1+m}b')

oracle = FeistelOracle().build_circuit(f1, f2, alpha0, alpha1, m)
measurements = simon.collect_measurements(oracle, n=1+m, backend=backend, shots=500)
s = simon.solve_period(measurements, n=1+m)

print(f"block size: {2*m}  (paper validates block sizes 6, 8 on real hardware)")
print(f"true s=(1,gamma):  {true_s}")
print(f"recovered:         {s}")
print(f"MATCH: {s == true_s}")


block size: 6  (paper validates block sizes 6, 8 on real hardware)
true s=(1,gamma):  1111
recovered:         1111
MATCH: True


## Summary vs. the paper's claims

| Attack | Paper's claim | This notebook (noiseless sim) |
|---|---|---|
| Bernstein-Vazirani | Single-query secret recovery, n=16, real hardware | ✓ single-query recovery, n=8 (sim) |
| Grover SPN | Quadratic speedup, n=8, ~13 iterations | ✓ matches iteration count and recovers key |
| Simon - Even-Mansour | Clean to N=5, hybrid (withheld technique) to N=10 | ✓ clean to N=8 (sim); N=6-10 needs the withheld technique |
| Simon - CBC-MAC forgery | Period s=Ek(a)^Ek(b) | Period is actually a^b (verified discrepancy) |
| Simon - 3-round Feistel | Clean at block 6, 8 (real hardware) | ✓ clean at block 6, 8, 10, 12 (sim) |

The disclosed algorithms are correctly implemented and validated end-to-end.
The paper's specific real-hardware noise-scaling claim (N=6-10) is out of
scope without the withheld technique -- see `README.md` and
`comparison/` for the full Stage 6 writeup.
